## 2. Python을 통한 API 호출 (`requests`)

### API 설명
<img src="https://velog.velcdn.com/images/newnew_daddy/post/23333b0c-a177-4782-b3de-0a3280de1f04/image.png" width="90%">

[Open Brewery DB 공식 문서](https://www.openbrewerydb.org/documentation)의 REST API를 GET으로 호출합니다.

- `requests.get(url)` → HTTP 응답
- `response.json()` → JSON을 **파이썬 리스트(또는 딕셔너리)** 로 변환
- `pd.json_normalize()` → 중첩된 dict 리스트를 **DataFrame 테이블** 로 변환


### 2-1. 기본 GET — 맥주 양조장 목록


In [1]:
import requests
import pandas as pd

url = "https://api.openbrewerydb.org/v1/breweries"
response = requests.get(url)

response.status_code, response.headers.get("Content-Type")

(200, 'application/json')

In [2]:
# JSON을 파이썬 객체로 변환 → 이 API는 양조장 dict의 **리스트** 를 반환합니다.
breweries = response.json()

type(breweries), len(breweries)

(list, 50)

In [3]:
# 리스트의 첫 번째 원소(dict) 확인
breweries[0]

{'id': 'ae7b3174-8be8-4d53-a3a5-9b8240970eea',
 'name': "'s",
 'brewery_type': 'brewpub',
 'address_1': 'Friesener Straße 1',
 'address_2': None,
 'address_3': None,
 'city': 'Kronach',
 'state_province': 'Bayern',
 'postal_code': '96317',
 'country': 'Germany',
 'longitude': 11.327765,
 'latitude': 50.241246,
 'phone': '+49 9261 628000',
 'website_url': 'http://www.antla.de',
 'state': 'Bayern',
 'street': 'Friesener Straße 1'}

### 2-2. `json_normalize`로 테이블 만들기
- `pd.json_normalize()` 는 dict 리스트를 평탄화해서 컬럼으로 펼칩니다.


In [4]:
df = pd.json_normalize(breweries)

df.head()

,id,name,brewery_type,address_1,address_2,address_3,city,state_province,postal_code,country,longitude,latitude,phone,website_url,state,street
0,ae7b3174-8be8-4d53-a3a5-9b8240970eea,'s,brewpub,Friesener Straße 1,None,None,Kronach,Bayern,96317,Germany,11.327765,50.241246,+49 9261 628000,http://www.antla.de,Bayern,Friesener Straße 1
1,aa7cbe9b-3a0f-4888-9884-6186b0042b55,’t Drankorgel,micro,31a Geelsebaan,None,None,Mol,Vlaanderen,2400,Belgium,5.056955,51.176258,+32 496 76 40 63,http://tdrankorgel.be/,Vlaanderen,31a Geelsebaan
2,1034754f-abf9-4b42-bd6d-37e110ba4b1a,'T Kroontje,micro,62 Hogebrug,None,None,Lebbeke,Vlaanderen,9280,Belgium,4.091393,51.007888,+32 495 43 33 25,http://www.tkroontje.be/,Vlaanderen,62 Hogebrug
3,907826b3-6902-4d28-a5e2-f73084bebeed,'t Veense Brouwhuys,micro,Langeweg 16,None,None,Roelofarendsveen,Zuid-Holland,2371 EG,Netherlands,NaN,NaN,071-3314585,https://www.t-veen.nl/t-veense-brouwhuys/,Zuid-Holland,Langeweg 16
4,5128df48-79fc-4f0f-8b52-d06be54d0cec,(405) Brewing Co,micro,1716 Topeka St,None,None,Norman,Oklahoma,73069-8224,United States,-97.468182,35.257389,4058160490,http://www.405brewing.com,Oklahoma,1716 Topeka St


### 2-3. 쿼리 파라미터로 필터링

`GET https://api.openbrewerydb.org/v1/breweries` 에 `params` 를 넘기면 문서의 필터를 그대로 쓸 수 있습니다.

| 파라미터 | 설명 |
|---|---|
| `per_page` | 페이지당 개수 (기본 50, 최대 200) |
| `page` | 페이지 번호 |
| `by_city` | 도시명 (공백은 `_` 또는 URL 인코딩) |
| `by_country` | 국가명 |
| `by_type` | `micro`, `nano`, `regional`, `brewpub` 등 |
| `by_name` | 양조장 이름 필터 |


#### 1) 개수 줄여서 가져오기 (`per_page`)


In [5]:
url = "https://api.openbrewerydb.org/v1/breweries"
params = {"per_page": 5}

data = requests.get(url, params=params).json()
pd.json_normalize(data)

,id,name,brewery_type,address_1,address_2,address_3,city,state_province,postal_code,country,longitude,latitude,phone,website_url,state,street
0,ae7b3174-8be8-4d53-a3a5-9b8240970eea,'s,brewpub,Friesener Straße 1,None,None,Kronach,Bayern,96317,Germany,11.327765,50.241246,+49 9261 628000,http://www.antla.de,Bayern,Friesener Straße 1
1,aa7cbe9b-3a0f-4888-9884-6186b0042b55,’t Drankorgel,micro,31a Geelsebaan,None,None,Mol,Vlaanderen,2400,Belgium,5.056955,51.176258,+32 496 76 40 63,http://tdrankorgel.be/,Vlaanderen,31a Geelsebaan
2,1034754f-abf9-4b42-bd6d-37e110ba4b1a,'T Kroontje,micro,62 Hogebrug,None,None,Lebbeke,Vlaanderen,9280,Belgium,4.091393,51.007888,+32 495 43 33 25,http://www.tkroontje.be/,Vlaanderen,62 Hogebrug
3,907826b3-6902-4d28-a5e2-f73084bebeed,'t Veense Brouwhuys,micro,Langeweg 16,None,None,Roelofarendsveen,Zuid-Holland,2371 EG,Netherlands,NaN,NaN,071-3314585,https://www.t-veen.nl/t-veense-brouwhuys/,Zuid-Holland,Langeweg 16
4,5128df48-79fc-4f0f-8b52-d06be54d0cec,(405) Brewing Co,micro,1716 Topeka St,None,None,Norman,Oklahoma,73069-8224,United States,-97.468182,35.257389,4058160490,http://www.405brewing.com,Oklahoma,1716 Topeka St


#### 2) 도시로 필터 (`by_city`)


In [6]:
params = {
    "by_city": "san_diego",
    "per_page": 10
}

data = requests.get(url, params=params).json()
pd.json_normalize(data)[["name", "brewery_type", "city", "state_province", "country"]]

,name,brewery_type,city,state_province,country
0,10 Barrel Brewing Co,large,San Diego,California,United States
1,2Kids Brewing Company,micro,San Diego,California,United States
2,32 North Brewing Co,brewpub,San Diego,California,United States
3,Abnormal Beer Company,micro,San Diego,California,United States
4,Acoustic Ales Brewing Experiment,closed,San Diego,California,United States
5,AleSmith Brewing Co,regional,San Diego,California,United States
6,Align Brewing Co,micro,San Diego,California,United States
7,Alpine Beer Company,brewpub,San Diego,California,United States
8,Alta Brewing Company,micro,San Diego,California,United States
9,Amplified Ale Works,brewpub,San Diego,California,United States


#### 3) 타입으로 필터 (`by_type`)


In [7]:
params = {
    "by_type": "micro",
    "per_page": 10
}

data = requests.get(url, params=params).json()
pd.json_normalize(data)[["name", "brewery_type", "city", "country"]]

,name,brewery_type,city,country
0,’t Drankorgel,micro,Mol,Belgium
1,'T Kroontje,micro,Lebbeke,Belgium
2,'t Veense Brouwhuys,micro,Roelofarendsveen,Netherlands
3,(405) Brewing Co,micro,Norman,United States
4,(512) Brewing Co,micro,Austin,United States
5,1 of Us Brewing Company,micro,Mount Pleasant,United States
6,10 Torr Distilling and Brewing,micro,Reno,United States
7,10-56 Brewing Company,micro,Knox,United States
8,100 Watt Brewery,micro,Eindhoven,Netherlands
9,105 West Brewing Co,micro,Castle Rock,United States


#### 4) 이름으로 검색 (`/search`)

부분 일치, 대소문자 무시. 결과가 없으면 빈 리스트 `[]` 입니다.


In [8]:
search_url = "https://api.openbrewerydb.org/v1/breweries/search"
params = {"query": "dog", "per_page": 10}

data = requests.get(search_url, params=params).json()
pd.json_normalize(data)[["name", "brewery_type", "city", "country"]]

,name,brewery_type,city,country
0,Dog & Pony Alehouse and Grill,closed,Renton,United States
1,Dog and Pony Brewing Company,taproom,Cashmere,United States
2,Dog Days Brewing,brewpub,Bremerton,United States
3,Dog Tag Brewing,contract,Bozeman,United States
4,Dog Money Restaurant,brewpub,Leesburg,United States
5,Dog Rose Brewing Company,micro,Saint Augustine,United States
6,Dogfish Head Craft Brewery,regional,Milton,United States
7,Dogfish Head Brewings & Eats,brewpub,Rehoboth Beach,United States
8,DogBerry Brewing LLC,micro,West Chester,United States
9,Dogtown Brewing,brewpub,Richmond,United States


#### 5) 랜덤 양조장 (`/random`)

- 기본은 양조장 **1개(dict)**
- `size` 를 2 이상으로 주면 **리스트**


In [9]:
random_url = "https://api.openbrewerydb.org/v1/breweries/random"

one = requests.get(random_url).json()
type(one), one

(list,
 [{'id': '425d365e-db70-46c4-be09-d000727eb003',
   'name': 'Cult Classic Brewing',
   'brewery_type': 'micro',
   'address_1': '1169 Shopping Center Drive',
   'address_2': None,
   'address_3': None,
   'city': 'Stevensville',
   'state_province': 'Maryland',
   'postal_code': '21666',
   'country': 'United States',
   'longitude': -76.2978846,
   'latitude': 38.9757006,
   'phone': '4109808097',
   'website_url': 'http://www.cultclassicbrewing.com',
   'state': 'Maryland',
   'street': '1169 Shopping Center Drive'}])

In [10]:
many = requests.get(random_url, params={"size": 5}).json()
pd.json_normalize(many)[["id", "name", "city", "country"]]

,id,name,city,country
0,fc2ce5e0-8796-4749-9aba-8710cd557ccb,Aqueduct Brewing,Akron,United States
1,1f51097a-5604-45ea-a694-76f816f3e248,White Bay Beer Co,Rozelle,Australia
2,eb91ff82-129b-4177-82f9-7646a4a1bb33,Montross Brewery,Montross,United States
3,9749c8f4-5273-449f-9563-0dcd12a14e42,Cape Ann Brewing Co,Gloucester,United States
4,6a01fef8-dd06-4ba6-99f8-fca4874e14af,Gristworkz,Houston,United States


#### 6) 단건 조회 (`/breweries/{id}`)

목록에서 얻은 `id`로 한 건만 가져옵니다. 응답은 **리스트가 아니라 dict** 이므로 `json_normalize`에 리스트로 감싸 줍니다.


In [11]:
brewery_id = df.loc[0, "id"]
detail_url = f"https://api.openbrewerydb.org/v1/breweries/{brewery_id}"

one = requests.get(detail_url).json()
type(one)

pd.json_normalize([one])

,id,name,brewery_type,address_1,address_2,address_3,city,state_province,postal_code,country,longitude,latitude,phone,website_url,state,street
0,ae7b3174-8be8-4d53-a3a5-9b8240970eea,'s,brewpub,Friesener Straße 1,None,None,Kronach,Bayern,96317,Germany,11.327765,50.241246,+49 9261 628000,http://www.antla.de,Bayern,Friesener Straße 1
